# Interactive SHAP Analysis for PatchXFormer

This notebook provides an interactive way to explore SHAP analysis results for your solar power forecasting model.

## Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import torch
import warnings

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline

print("Libraries loaded successfully!")

## Load Pre-computed SHAP Results

If you've already run the SHAP analysis, load the results here:

In [ ]:
# Load feature importance
results_dir = 'results'  # Update this path

importance_df = pd.read_csv(f'{results_dir}/csv_reports/feature_importance_weather.csv')
print("Feature Importance Rankings:")
print(importance_df)

In [ ]:
# Load detailed SHAP values
shap_detailed = pd.read_csv(f'{results_dir}/csv_reports/shap_values_detailed.csv')
print(f"Loaded {len(shap_detailed)} samples with SHAP values")
print("\nColumns:")
print(shap_detailed.columns.tolist())

## Visualize Feature Importance

In [ ]:
# Create bar plot
plt.figure(figsize=(10, 6))
bars = plt.barh(importance_df['Feature'], importance_df['Mean_Abs_SHAP'])

# Color code by importance
colors = plt.cm.RdYlGn_r(importance_df['Mean_Abs_SHAP'] / importance_df['Mean_Abs_SHAP'].max())
for bar, color in zip(bars, colors):
    bar.set_color(color)

plt.xlabel('Mean Absolute SHAP Value', fontsize=12, fontweight='bold')
plt.ylabel('Weather Feature', fontsize=12, fontweight='bold')
plt.title('Weather Feature Importance for Solar Power Forecasting', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Analyze Top Features

In [ ]:
# Extract top 3 features
top_features = importance_df.head(3)['Feature'].tolist()
print(f"Top 3 most important features: {top_features}")

# Create scatter plots for top features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(top_features):
    ax = axes[idx]
    
    # Get feature values and SHAP values
    feature_values = shap_detailed[f'Value_{feature}']
    shap_values = shap_detailed[f'SHAP_{feature}']
    
    # Create scatter plot
    scatter = ax.scatter(feature_values, shap_values, 
                        c=feature_values, cmap='viridis', alpha=0.6, s=30)
    
    # Add trend line
    z = np.polyfit(feature_values, shap_values, 2)
    p = np.poly1d(z)
    x_trend = np.linspace(feature_values.min(), feature_values.max(), 100)
    ax.plot(x_trend, p(x_trend), "r--", linewidth=2, alpha=0.8)
    
    ax.set_xlabel(f'{feature} Value', fontsize=11, fontweight='bold')
    ax.set_ylabel('SHAP Value', fontsize=11, fontweight='bold')
    ax.set_title(f'Impact of {feature}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax)

plt.tight_layout()
plt.show()

## Statistical Analysis

In [ ]:
from scipy.stats import pearsonr

# Analyze correlation between feature values and SHAP values
print("Feature-SHAP Correlations:\n")
print("-" * 60)

for feature in importance_df['Feature']:
    feature_vals = shap_detailed[f'Value_{feature}']
    shap_vals = shap_detailed[f'SHAP_{feature}']
    
    corr, p_value = pearsonr(feature_vals, shap_vals)
    
    print(f"{feature:15s}: r = {corr:7.4f}, p = {p_value:.2e}")
    
    if corr > 0:
        print(f"                → Higher {feature} → Higher solar power")
    else:
        print(f"                → Higher {feature} → Lower solar power")
    print()

## Feature Interaction Analysis

In [ ]:
# Load correlation matrix
correlation_df = pd.read_csv(f'{results_dir}/csv_reports/feature_shap_correlations.csv', index_col=0)

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_df, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('SHAP Value Correlations: Feature Interactions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Find strongest interactions
corr_matrix = correlation_df.values
np.fill_diagonal(corr_matrix, 0)
max_idx = np.unravel_index(np.abs(corr_matrix).argmax(), corr_matrix.shape)
feat1 = correlation_df.index[max_idx[0]]
feat2 = correlation_df.columns[max_idx[1]]
max_corr = corr_matrix[max_idx]

print(f"\nStrongest feature interaction:")
print(f"{feat1} ↔ {feat2}: r = {max_corr:.4f}")

## Distribution Analysis

In [ ]:
# Plot distributions of SHAP values
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(importance_df['Feature']):
    ax = axes[idx]
    shap_vals = shap_detailed[f'SHAP_{feature}']
    
    ax.hist(shap_vals, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(shap_vals.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
    ax.axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
    
    ax.set_xlabel('SHAP Value', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{feature}', fontsize=11, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Hide unused subplot
axes[-1].axis('off')

plt.suptitle('Distribution of SHAP Values for Each Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
# Calculate comprehensive statistics
stats_list = []

for feature in importance_df['Feature']:
    shap_vals = shap_detailed[f'SHAP_{feature}']
    
    stats = {
        'Feature': feature,
        'Mean': shap_vals.mean(),
        'Std': shap_vals.std(),
        'Min': shap_vals.min(),
        'Q25': shap_vals.quantile(0.25),
        'Median': shap_vals.median(),
        'Q75': shap_vals.quantile(0.75),
        'Max': shap_vals.max(),
        'Mean_Abs': np.abs(shap_vals).mean()
    }
    stats_list.append(stats)

stats_df = pd.DataFrame(stats_list)
stats_df = stats_df.sort_values('Mean_Abs', ascending=False)

print("SHAP Value Statistics by Feature:\n")
print(stats_df.to_string(index=False))

## Key Insights for Thesis

Run this cell to generate a summary of key findings:

In [ ]:
print("="*80)
print("KEY FINDINGS FOR THESIS")
print("="*80)
print()

print("1. MOST IMPORTANT WEATHER VARIABLES:")
print("-" * 40)
for idx, row in importance_df.head(3).iterrows():
    print(f"   {row['Importance_Rank']}. {row['Feature']}: "
          f"Mean |SHAP| = {row['Mean_Abs_SHAP']:.6f}")
print()

print("2. DIRECTIONAL IMPACTS:")
print("-" * 40)
for feature in importance_df['Feature']:
    shap_mean = shap_detailed[f'SHAP_{feature}'].mean()
    direction = "increases" if shap_mean > 0 else "decreases"
    print(f"   • Higher {feature} {direction} predicted solar power")
print()

print("3. FEATURE STABILITY:")
print("-" * 40)
for idx, row in stats_df.iterrows():
    cv = row['Std'] / (abs(row['Mean']) + 1e-10)  # Coefficient of variation
    stability = "stable" if cv < 1 else "variable"
    print(f"   • {row['Feature']}: {stability} (CV = {cv:.2f})")
print()

print("="*80)

## Export Results for Thesis

Generate formatted tables for your thesis:

In [ ]:
# Format for LaTeX table
print("LaTeX Table: Feature Importance\n")
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Weather Feature Importance for Solar Power Forecasting using SHAP Analysis}")
print("\\begin{tabular}{clr}")
print("\\hline")
print("\\textbf{Rank} & \\textbf{Feature} & \\textbf{Mean |SHAP|} \\\\")
print("\\hline")

for idx, row in importance_df.iterrows():
    print(f"{row['Importance_Rank']} & {row['Feature']} & {row['Mean_Abs_SHAP']:.4f} \\\\")

print("\\hline")
print("\\end{tabular}")
print("\\label{tab:shap_importance}")
print("\\end{table}")

## Conclusion

This notebook provided an interactive exploration of SHAP analysis results. Key outputs:

1. Feature importance rankings
2. Directional impacts of features
3. Feature interactions
4. Statistical distributions
5. Formatted tables for thesis

Use these insights to:
- Understand which weather variables drive predictions
- Improve model by focusing on important features
- Write interpretability section of thesis
- Compare with other models (e.g., CatBoost from reference paper)